# judge-audit · independent re-score on a Kaggle T4

Re-trains every kept commit of one condition in a fresh worktree, with the stock harness restored and a clean data cache, and scores each model on the hidden shard. The unmodified baseline is re-run `BASE_REPEATS` times first to measure run-to-run noise.

**Why on a T4:** `train.py` stops on a 120 s wall clock, so the step count (and `val_bpb`) depends on the GPU. The re-score must run on the hardware the agent used.

**Settings:** Accelerator = GPU T4 x2 · Internet = On · **no secrets** (there is no agent in this run) · Input = your private dataset holding `agent-<C>.bundle` and `audit-env-<C>.tgz` from the condition run.

**Run it with Save Version → Save & Run All (Commit).** About 10 training runs of ~4 min each, so ~45–60 min of GPU quota.

## Cell 1: run the re-score

- `BASE` is the run's baseline commit; `TIP` is the **last kept** commit (for run A not the branch head `e699b0c`, which was never logged).
- Must be `!` lines, not `%%bash`, so the version keeps its outputs even if the script fails.

In [ ]:
CONDITION = "A"
BASE = "fbf5e34"       # baseline commit (run card)
TIP = "dc153d6"        # last kept commit (run card)
BASE_REPEATS = 3
!rm -rf /kaggle/working/judge-audit
!git clone https://github.com/baderddine725/judge-audit.git /kaggle/working/judge-audit
!BASE={BASE} TIP={TIP} BASE_REPEATS={BASE_REPEATS} bash /kaggle/working/judge-audit/setup/kaggle_audit.sh {CONDITION}

## Outputs (Output tab of the committed version)

| File | What it is |
|---|---|
| `reeval-<C>.tsv` | one row per run: re-run's own `val_bpb`, hidden-shard bpb, kind (`base` repeat or `kept`) |
| `reeval-<C>.log` | full console log, including the harness diff and environment checks |
| `reeval-logs-<C>.tgz` | each run's training log |

Copy `reeval-<C>.tsv` to `runs/<C>/reeval.tsv` and score it with `audit/score.py --reeval`.